# Explainable Boosting Machine (EBM)

In [1]:
import os

print("=== TEST TMUX ===")
tmux_var = os.environ.get("TMUX")

if tmux_var:
    print("STATUS: Jupyter server is running in a TMUX session!")
    print(f"Tmux socket path: {tmux_var}")
else:
    print("STATUS: The Jupyter server is NOT inside Tmux.")

=== TEST TMUX ===
STATUS: Jupyter server is running in a TMUX session!
Tmux socket path: /tmp/tmux-1022/default,126489,1


## Data extraction

In [2]:
from utils.data_processing import load_and_preprocess_data

In [3]:
X, y, feature_names = load_and_preprocess_data(dataset_path="../Data/MARSIS_historical_dataset.csv", orbit_path="../Data/orbit_to_remove", keep_flux=True)

### EBM Experiment 1

In [ ]:
from interpret.glassbox import ExplainableBoostingRegressor
from utils.pipeline import pipeline

seed_exp_1 = 42

ebm_parameters = {
    "interactions": 0,
    "outer_bags": 4,
    "max_bins": 128
}

k_fold_dict_ebm, save_dir_ebm = pipeline(
    X=X, 
    y=y, 
    model_class=ExplainableBoostingRegressor, # passing Regressor class
    n_splits=10, 
    exp_name="ebm_experiment_1",
    seed_ebm=seed_exp_1,
    model_kwargs=ebm_parameters
)

=== Starting Experiment Pipeline: ebm_experiment_1 ===
Created isolated environment at: experiments/ebm_experiment_1_20260724_203601

--- Starting FOLD 0 ---
Train MSE: 7.816, Test MSE: 27.117, Train MAE: 2.135, Test MAE: 4.120

--- Starting FOLD 1 ---
Train MSE: 7.805, Test MSE: 22.069, Train MAE: 2.117, Test MAE: 3.773

--- Starting FOLD 2 ---
Train MSE: 7.808, Test MSE: 21.159, Train MAE: 2.121, Test MAE: 3.723

--- Starting FOLD 3 ---
Train MSE: 7.726, Test MSE: 19.567, Train MAE: 2.109, Test MAE: 3.560

--- Starting FOLD 4 ---

Execution 'ebm_experiment_1' successfully completed in 48737.11s!
Generating Learning Curves...
Note: Explainable Boosting Machine detected. Skipping Learning Curves plotting.
Experiment completely saved in: /home/emiliano/projects/project_1/Lab_XAI/Lab_XAI_pytorch/experiments/ebm_experiment_1_20260724_203601


### EBM Explanation Plots Experiment 1

#### Global explanation

In [4]:
import os
import pandas as pd
from joblib import load
from interpret import show


model_path_exp_1 = "experiments/ebm_experiment_1_20260724_203601/model_ebm_experiment_1_5.save"

# loading model
ebm_model_exp_1 = load(model_path_exp_1)
print("Loaded model successfully!")

Loaded model successfully!


In [5]:
from datetime import datetime

plots_dir = "plots"
os.makedirs(plots_dir, exist_ok=True)
timestamp_1 = datetime.now().strftime("%Y%m%d_%H%M%S")
# Inject real names into the internal attribute of the loaded model
ebm_model_exp_1.feature_names_ = feature_names

# If there are strings/categories in the dataset, EBM may want to update 
# the display names of the relationships as well. To be safe, let's update this as well:
ebm_model_exp_1.term_names_ = feature_names

# Generates the global explanation (without parameters, it will use the ones just injected)
ebm_global_exp_1 = ebm_model_exp_1.explain_global()

# Chart 1: Global Feature Importance (Summary)
print("Loading Feature Importance Graph...")
fig_summary_1 = ebm_global_exp_1.visualize()  # without inputs it plots global summary
path_summary_png_1 = os.path.join(plots_dir, f"EBM_Feature_Importance_{timestamp_1}.png")
path_summary_pdf_1 = os.path.join(plots_dir, f"EBM_Feature_Importance_{timestamp_1}.pdf")
fig_summary_1.write_image(path_summary_png_1, scale=3, width=1000, height=700) 
fig_summary_1.write_image(path_summary_pdf_1, width=1000, height=700)
fig_summary_1.show()


Loading Feature Importance Graph...


#### Local explanation - data altitude

In [6]:
# Chart 2: The curve (Shape Function) of a specific variable

# variable to be analyzed
name_feat_2 = "FM_data_altitude" 
timestamp_2 = datetime.now().strftime("%Y%m%d_%H%M%S")

# finding index of such variable in the list
feature_index_2 = feature_names.index(name_feat_2)

print(f"Loading curve for feature: {name_feat_2} (Positional index: {feature_index_2})...")

# passes entire index to visualization function
fig_feature_2 = ebm_global_exp_1.visualize(feature_index_2)
path_feat_png_2 = os.path.join(plots_dir, f"EBM_shape_{name_feat_2}_{timestamp_2}.png")
path_feat_pdf_2 = os.path.join(plots_dir, f"EBM_shape_{name_feat_2}_{timestamp_2}.pdf")

fig_feature_2.write_image(path_feat_png_2, scale=3, width=900, height=600)
fig_feature_2.write_image(path_feat_pdf_2, width=900, height=600)
fig_feature_2.show()

Loading curve for feature: FM_data_altitude (Positional index: 0)...


#### Local explanation - solar flux

In [7]:
# Chart 3: The curve (Shape Function) of a specific variable

# variable to be analyzed
name_feat_3 = "FM_data_F10_7_index" 
timestamp_3 = datetime.now().strftime("%Y%m%d_%H%M%S")

# finding index of such variable in the list
feature_3_index = feature_names.index(name_feat_3)

print(f"Loading curve for feature: {name_feat_3} (Positional index: {feature_3_index})...")

# passes entire index to visualization function
fig_feature_3 = ebm_global_exp_1.visualize(feature_3_index)
path_feat3_png = os.path.join(plots_dir, f"EBM_shape_{name_feat_3}_{timestamp_3}.png")
path_feat3_pdf = os.path.join(plots_dir, f"EBM_shape_{name_feat_3}_{timestamp_3}.pdf")

fig_feature_3.write_image(path_feat3_png, scale=3, width=900, height=600)
fig_feature_3.write_image(path_feat3_pdf, width=900, height=600)
fig_feature_3.show()

Loading curve for feature: FM_data_F10_7_index (Positional index: 3)...
